- Load and prepare the wildfire dataset 
- define chronological train/validation/test splits
- reserve 2020 as an independent stress-test year 
- verify dataset size and class balance.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
 
warnings.filterwarnings("ignore")
 
RANDOM_STATE = 42
TARGET       = "next_day_risk_class"
OUT_DIR      = Path("../data/training/engineered")
PLOT_DIR     = Path("outputs")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
 
df = pd.read_parquet("../data/training/integrated/algeria_wildfire_dataset.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["commune_id", "date"]).reset_index(drop=True)
df["year"] = df["date"].dt.year
 
TRAIN_YEARS  = [2015, 2016, 2017, 2018]
VAL_YEARS    = [2019]
TEST_YEARS   = [2021, 2022, 2023, 2024, 2025]
STRESS_YEARS = [2020]
 
train_mask  = df["year"].isin(TRAIN_YEARS)
val_mask    = df["year"].isin(VAL_YEARS)
test_mask   = df["year"].isin(TEST_YEARS)
stress_mask = df["year"].isin(STRESS_YEARS)
 
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df.date.min().date()} -> {df.date.max().date()}")
print(f"Communes: {df.commune_id.nunique():,}")
for name, mask in [("TRAIN",train_mask),("VAL",val_mask),("TEST",test_mask),("STRESS-2020",stress_mask)]:
    pos = 100*(df.loc[mask, TARGET]>0).mean()
    print(f"  {name:<12} rows={mask.sum():>7,} | positive={pos:.1f}%")

Loaded: 157,960 rows x 42 columns
Date range: 2015-05-01 -> 2025-10-30
Communes: 1,245
  TRAIN        rows= 56,763 | positive=19.0%
  VAL          rows= 15,717 | positive=28.1%
  TEST         rows= 68,581 | positive=15.7%
  STRESS-2020  rows= 16,899 | positive=33.1%


### Load Dense Calendar

- Load and parse the dense calendar.
- Extract `month` and sort by `commune_id`, `season_year`, and `date`.
- The dense calendar includes downsampled negatives needed for correct calendar-based feature computation.
- `df` is only a sampled subset used for modeling.

In [2]:
dense = pd.read_parquet("../data/training/integrated/algeria_wildfire_dense_calendar.parquet")
dense["date"]  = pd.to_datetime(dense["date"])
dense["month"] = dense["date"].dt.month
dense = dense.sort_values(["commune_id", "season_year", "date"]).reset_index(drop=True)

print(f"Dense calendar loaded: {dense.shape[0]:,} rows x {dense.shape[1]} columns "
      f"({dense.commune_id.nunique():,} communes)")
print(f"Sampled modeling table df: {df.shape[0]:,} rows "
      f"({100*df.shape[0]/dense.shape[0]:.1f}% of the dense calendar — "
      f"the rest are downsampled negatives, correctly excluded from training "
      f"but needed here to compute correct calendar-based features)")


Dense calendar loaded: 2,506,185 rows x 15 columns (1,245 communes)
Sampled modeling table df: 157,960 rows (6.3% of the dense calendar — the rest are downsampled negatives, correctly excluded from training but needed here to compute correct calendar-based features)


## Drop Redundant Raw Features

- Justified by EDA correlation matrix:
  - **NDWI:** r=0.92 with NDVI, r=0.98 with NBR → fully redundant
  - **BUI:** r=0.99 with DMC → mathematically near-identical
  - **ISI:** r=0.92 with FWI → FWI already integrates ISI

- Other low-value columns:
  - **wind_dir, aspect_mean_deg:** no monotonic fire relationship
  - **wind_speed_kmh:** r=-0.006 with target, encoded in FWI already
  - **shrub_fraction, crop_fraction, grass_fraction:** burnable_fraction already aggregates them; keeping all adds noise without gain

In [3]:
DROP_COLS = [
    "NDWI", "BUI", "ISI",
    "wind_dir", "aspect_mean_deg", "wind_speed_kmh",
    "shrub_fraction", "crop_fraction", "grass_fraction",
]
drop_present = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=drop_present)
print(f"Dropped {len(drop_present)}: {drop_present}")
print(f"Columns remaining: {df.shape[1]}")

Dropped 9: ['NDWI', 'BUI', 'ISI', 'wind_dir', 'aspect_mean_deg', 'wind_speed_kmh', 'shrub_fraction', 'crop_fraction', 'grass_fraction']
Columns remaining: 33


### Fire Persistence Features

- Fill missing fire values with `0` and compute:
  - `frp_total` = fire count × mean FRP
  - `fire_active` = whether a fire is present
- Rebuild `frp_total` on the **dense calendar**.
- Compute rolling fire/FRP features within each `(commune_id, season_year)` to avoid crossing fire-season boundaries.
- Compute `days_since_fire`, resetting at each season.
- Compute `wilaya_fire_excl_self` as the wilaya fire activity excluding the commune itself.
- Merge these dense-calendar features back into the sampled `df`.

In [4]:
df["fire_count"] = df["fire_count"].fillna(0)
df["mean_frp"]   = df["mean_frp"].fillna(0)
df["frp_total"]  = df["fire_count"] * df["mean_frp"]  # intensity interaction
df["fire_active"] = (df["fire_count"] > 0).astype("int8")

# frp_total isn't on the dense calendar yet 
dense["mean_frp"]  = np.where(
    dense["fire_count"] > 0,
    dense["frp_sum"] / dense["fire_count"].clip(lower=1),
    0.0
)
dense["frp_total"] = dense["fire_count"] * dense["mean_frp"]

def rolling_sum(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=1).sum()

def rolling_mean(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=1).mean()

# Grouped by (commune_id, season_year)
dgrp = dense.groupby(["commune_id", "season_year"], group_keys=False)

dense["fire_count_3d"] = dgrp.apply(lambda g: rolling_sum(g, "fire_count", 3)).values
dense["fire_count_7d"] = dgrp.apply(lambda g: rolling_sum(g, "fire_count", 7)).values
dense["frp_total_7d"]  = dgrp.apply(lambda g: rolling_sum(g, "frp_total",  7)).values

# days_since_fire — same grouping, so the counter resets at each season's
# start instead of carrying a stale count over from last October.
def days_since(series):
    result, count = [], 0
    for val in series:
        count = 0 if val > 0 else count + 1
        result.append(count)
    return pd.Series(result, index=series.index)

dense["days_since_fire"] = (
    dense.groupby(["commune_id", "season_year"])["fire_count"].transform(days_since)
)

# Wilaya spatial lag — wilaya_fire_sum (built in build_dataset.py) already
# sums fire_count across ALL fire-prone communes in the wilaya for that
# date, using the dense grid, not the sampled skeleton.
dense["wilaya_fire_excl_self"] = (
    dense["wilaya_fire_sum"] - dense["fire_count"]
).clip(lower=0)

# Merge the dense-computed features onto the sampled modeling table
fire_hist_cols = ["fire_count_3d", "fire_count_7d", "frp_total_7d",
                   "days_since_fire", "wilaya_fire_excl_self"]
df = df.merge(
    dense[["commune_id", "date"] + fire_hist_cols],
    on=["commune_id", "date"], how="left"
)

print("Fire persistence features (computed on the dense calendar, joined onto df):")
for col in fire_hist_cols:
    print(f"  {col}: mean={df[col].mean():.2f}  max={df[col].max():.1f}")


Fire persistence features (computed on the dense calendar, joined onto df):
  fire_count_3d: mean=0.48  max=510.0
  fire_count_7d: mean=0.91  max=674.0
  frp_total_7d: mean=13.67  max=15417.3
  days_since_fire: mean=58.52  max=183.0
  wilaya_fire_excl_self: mean=3.82  max=3112.0


### Historical Commune Fire Rate

- Compute each commune’s historical fire rate on the **dense calendar**, using training years only.
- Merge `commune_fire_rate` into the sampled `df`.
- Missing communes are filled with the **global training positive rate**.

In [5]:
# Computed on the DENSE calendar restricted to train years 
dense_train_mask = dense["season_year"].isin(TRAIN_YEARS)

train_fire_rate = (
    dense[dense_train_mask]
    .groupby("commune_id")["next_day_risk_class"]
    .apply(lambda x: (x > 0).mean())
    .reset_index()
    .rename(columns={"next_day_risk_class": "commune_fire_rate"})
)
global_rate = (dense.loc[dense_train_mask, "next_day_risk_class"] > 0).mean()

df = df.merge(train_fire_rate, on="commune_id", how="left")
df["commune_fire_rate"] = df["commune_fire_rate"].fillna(global_rate)

print(f"commune_fire_rate (true historical rate, from dense calendar): "
      f"mean={df.commune_fire_rate.mean():.3f}  "
      f"min={df.commune_fire_rate.min():.3f}  max={df.commune_fire_rate.max():.3f}")
print(f"Global train positive rate (fill value): {global_rate:.3f}")


commune_fire_rate (true historical rate, from dense calendar): mean=0.017  min=0.000  max=0.242
Global train positive rate (fill value): 0.012


### Weather History Features

- Compute 7-day historical weather features on the **dense calendar** within each `(commune_id, season_year)`.
- Features:
  - `FWI_7d`, `DC_7d`, `DMC_7d`: 7-day means
  - `precip_7d`: 7-day precipitation sum
  - `FWI_trend`: current FWI − 7-day mean FWI
- Merge these features into the sampled `df`.

In [6]:
dgrp = dense.groupby(["commune_id", "season_year"], group_keys=False)

dense["FWI_7d"]    = dgrp.apply(lambda g: rolling_mean(g, "FWI", 7)).values
dense["DC_7d"]     = dgrp.apply(lambda g: rolling_mean(g, "DC",  7)).values
dense["DMC_7d"]    = dgrp.apply(lambda g: rolling_mean(g, "DMC", 7)).values
dense["precip_7d"] = dgrp.apply(lambda g: rolling_sum(g,  "precip_mm", 7)).values
dense["FWI_trend"] = dense["FWI"] - dense["FWI_7d"]  # positive = worsening

weather_hist_cols = ["FWI_7d", "DC_7d", "DMC_7d", "precip_7d", "FWI_trend"]
df = df.merge(
    dense[["commune_id", "date"] + weather_hist_cols],
    on=["commune_id", "date"], how="left"
)

print(f"FWI_7d mean:    {df.FWI_7d.mean():.1f}")
print(f"DC_7d mean:     {df.DC_7d.mean():.1f}")
print(f"precip_7d mean: {df.precip_7d.mean():.2f} mm")
print(f"FWI_trend mean: {df.FWI_trend.mean():.2f}")


FWI_7d mean:    28.8
DC_7d mean:     540.2
precip_7d mean: 8.17 mm
FWI_trend mean: 0.61


### NBR & FFMC Anomalies

- Compute monthly **NBR baselines** per commune using training data and derive `NBR_anomaly`.
- Compute monthly **FFMC baselines** from the dense training calendar and derive `FFMC_anomaly`.
- Fill missing baselines with the corresponding global training mean.
- Drop temporary baseline columns after computing anomalies.
- Verify that the training `NBR_anomaly` is centered near zero.

In [7]:
train_ref = df[train_mask].copy()

# NBR
nbr_baseline = (
    train_ref.groupby(["commune_id", "month"])["NBR"]
    .mean().reset_index().rename(columns={"NBR": "_NBR_base"})
)
df = df.merge(nbr_baseline, on=["commune_id", "month"], how="left")
global_nbr_base = train_ref["NBR"].mean()
df["_NBR_base"] = df["_NBR_base"].fillna(global_nbr_base)
df["NBR_anomaly"] = df["NBR"] - df["_NBR_base"]
df = df.drop(columns=["_NBR_base"])

# FFMC
dense_train = dense[dense["season_year"].isin(TRAIN_YEARS)]
ffmc_baseline = (
    dense_train.groupby(["commune_id", "month"])["FFMC"]
    .mean().reset_index().rename(columns={"FFMC": "_FFMC_base"})
)
df = df.merge(ffmc_baseline, on=["commune_id", "month"], how="left")
global_ffmc_base = dense_train["FFMC"].mean()
df["_FFMC_base"] = df["_FFMC_base"].fillna(global_ffmc_base)
df["FFMC_anomaly"] = df["FFMC"] - df["_FFMC_base"]
df = df.drop(columns=["_FFMC_base"])

print(f"NBR_anomaly:   mean={df.NBR_anomaly.mean():.4f}  std={df.NBR_anomaly.std():.4f}")
print(f"FFMC_anomaly:  mean={df.FFMC_anomaly.mean():.4f}  std={df.FFMC_anomaly.std():.4f}")
assert abs(df.loc[train_mask, "NBR_anomaly"].mean()) < 0.01, "NBR anomaly mean not ~0 on train"


NBR_anomaly:   mean=-0.0031  std=0.0519
FFMC_anomaly:  mean=1.8687  std=10.6645


### Temporal Features

- Encode `month` and `day-of-year` as cyclical features using sine/cosine transformations.
- Features: `month_sin`, `month_cos`, `doy_sin`, `doy_cos`.

In [8]:
doy = df["date"].dt.dayofyear
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"]   = np.sin(2 * np.pi * doy / 365.25)
df["doy_cos"]   = np.cos(2 * np.pi * doy / 365.25)
 
print("Temporal features: month_sin, month_cos, doy_sin, doy_cos")

Temporal features: month_sin, month_cos, doy_sin, doy_cos


### Final Feature Set Definition

- Define the final feature set based on:
  1. EDA Pearson correlation with the target
  2. EDA mutual information scores
  3. Domain knowledge (fire behavior triangle: fuel, weather, terrain)
  4. Anti-redundancy (one representative per correlated group)
  5. Leakage safety (all features observable before prediction day)

- The final features cover:
  - Same-day fire state
  - Fire persistence and spread
  - Historical commune fire propensity
  - Weather and FWI drought memory
  - Vegetation conditions and anomalies
  - Terrain
  - Land cover
  - Human exposure
  - Temporal seasonality

- Verify that all selected features exist in the dataset before modeling.

In [9]:
FINAL_FEATURES = [
    # Same-day fire state (strongest predictors per EDA)
    "fire_count", "mean_frp", "frp_total", "fire_active",
 
    # Fire history — persistence and spread
    "fire_count_3d", "fire_count_7d", "frp_total_7d",
    "days_since_fire", "wilaya_fire_excl_self",
 
    # Commune baseline fire propensity (train-period historical rate)
    "commune_fire_rate",
 
    # Weather (EDA: temp r=0.183, rh r=-0.119)
    "temp_c", "rh", "precip_mm", "soil_moisture",
 
    # FWI system — drought memory (EDA: DC separation +118.87, DMC +67.68)
    "FFMC", "DMC", "DC", "FWI",
    "FWI_7d", "DC_7d", "DMC_7d", "precip_7d", "FWI_trend",
 
    # Vegetation (real signal after type==0 filter: NDVI r=0.148, NBR r=0.136)
    "NDVI", "NBR", "NBR_anomaly", "FFMC_anomaly",
 
    # Terrain (EDA MI: slope 0.119, elevation 0.120)
    "elevation_mean_m", "slope_mean_deg",
 
    # Land cover
    "burnable_fraction", "forest_fraction",
 
    # Human exposure
    "pop_density_mean", "road_distance_mean_km",
 
    # Temporal encoding
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]
 
# Verify all features exist
missing_feats = [f for f in FINAL_FEATURES if f not in df.columns]
if missing_feats:
    print(f" Missing features: {missing_feats}")
else:
    print(f"All {len(FINAL_FEATURES)} features present")
    print(f"Features: {FINAL_FEATURES}")

All 37 features present
Features: ['fire_count', 'mean_frp', 'frp_total', 'fire_active', 'fire_count_3d', 'fire_count_7d', 'frp_total_7d', 'days_since_fire', 'wilaya_fire_excl_self', 'commune_fire_rate', 'temp_c', 'rh', 'precip_mm', 'soil_moisture', 'FFMC', 'DMC', 'DC', 'FWI', 'FWI_7d', 'DC_7d', 'DMC_7d', 'precip_7d', 'FWI_trend', 'NDVI', 'NBR', 'NBR_anomaly', 'FFMC_anomaly', 'elevation_mean_m', 'slope_mean_deg', 'burnable_fraction', 'forest_fraction', 'pop_density_mean', 'road_distance_mean_km', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']


### Mutual Information Analysis

- Compute **Mutual Information (MI)** scores using the training set only.
- Rank `FINAL_FEATURES` by their MI with the target.
- Flag features with `MI < 0.003` for review.
- Low-MI features are retained when justified by domain knowledge.

In [10]:
X_mi = df.loc[train_mask, FINAL_FEATURES].fillna(0)
y_mi = df.loc[train_mask, TARGET]
 
mi_scores = mutual_info_classif(X_mi, y_mi, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({"feature": FINAL_FEATURES, "mi_score": mi_scores})
mi_df = mi_df.sort_values("mi_score", ascending=False).reset_index(drop=True)
 
print("Mutual Information scores (train only):")
print(mi_df.to_string(index=False))
 
# Flag any feature with MI < 0.003 that we are force-keeping
low_mi = mi_df[mi_df["mi_score"] < 0.003]
if len(low_mi):
    print(f"\nLow MI features (<0.003): {low_mi.feature.tolist()}")
    print("   These are kept for domain reasons — review if model performance is poor.")
else:
    print("\nAll features have MI > 0.003")

Mutual Information scores (train only):
              feature  mi_score
     elevation_mean_m  0.125093
       slope_mean_deg  0.124622
    commune_fire_rate  0.121541
     pop_density_mean  0.121297
road_distance_mean_km  0.120573
    burnable_fraction  0.116705
          NBR_anomaly  0.114752
      forest_fraction  0.110640
                 NDVI  0.094252
                  NBR  0.087717
      days_since_fire  0.085996
        fire_count_7d  0.072662
         frp_total_7d  0.070903
            frp_total  0.057368
           fire_count  0.057012
             mean_frp  0.053035
          fire_active  0.051676
        fire_count_3d  0.051015
wilaya_fire_excl_self  0.047086
              doy_sin  0.041375
              doy_cos  0.038548
               temp_c  0.033429
                   DC  0.032875
        soil_moisture  0.030118
                  FWI  0.028790
                  DMC  0.027023
                DC_7d  0.026775
            month_sin  0.025923
               DMC_7d  0.025918


### Missing Value Audit and Fill

- Audit the final raw feature sets for missing values.
- Handle missing rolling features caused by unavailable prior-day history.
- Use same-day values as a fallback where the corresponding base feature exists.
- Fill any remaining missing values with 0.
- Perform a final check to ensure no missing values remain before modeling.

In [11]:
missing = df[FINAL_FEATURES].isnull().sum()
missing = missing[missing > 0]

if len(missing):
    print("Missing values:")
    for col, n in missing.items():
        print(f"  {col}: {n:,} ({100*n/len(df):.1f}%)")

    # Rolling features: use same-day value when no prior history exists
    fallback_map = {
        "fire_count_3d": "fire_count",
        "fire_count_7d": "fire_count",
        "frp_total_7d": "frp_total",
        "FWI_7d": "FWI",
        "DC_7d": "DC",
        "DMC_7d": "DMC",
        "precip_7d": "precip_mm",
        "FWI_trend": "FWI",
    }

    for col, base in fallback_map.items():
        if col in missing.index:
            df[col] = df[col].fillna(df[base])

    # Remaining missing values
    df[FINAL_FEATURES] = df[FINAL_FEATURES].fillna(0)

# Final check
remaining = df[FINAL_FEATURES].isnull().sum().sum()
print(f"\nAfter fill: {remaining} missing values remaining")

assert remaining == 0, "Still has missing values"

Missing values:
  fire_count_3d: 718 (0.5%)
  fire_count_7d: 718 (0.5%)
  frp_total_7d: 718 (0.5%)
  FWI_7d: 718 (0.5%)
  DC_7d: 718 (0.5%)
  DMC_7d: 718 (0.5%)
  precip_7d: 718 (0.5%)
  FWI_trend: 718 (0.5%)

After fill: 0 missing values remaining


### Build Splits and Save

- Build the final modeling dataset using the selected features and target.
- Create chronologically separated train, validation, test, and 2020 stress-test splits.
- Verify class distributions and temporal separation between splits.
- Save the engineered dataset, individual splits, and feature metadata for modeling.

In [12]:
ID_COLS = ["date", "commune_id", "commune_name", "wilaya_id", "wilaya_name", "year"]
 
df_model     = df[ID_COLS + FINAL_FEATURES     + [TARGET]].copy()
 
train      = df_model[df_model["year"].isin(TRAIN_YEARS)].copy()
val        = df_model[df_model["year"].isin(VAL_YEARS)].copy()
test       = df_model[df_model["year"].isin(TEST_YEARS)].copy()
stress_20  = df_model[df_model["year"].isin(STRESS_YEARS)].copy()
 
for name, split in [("TRAIN",train),("VAL",val),("TEST",test),("STRESS-2020",stress_20)]:
    pos = 100*(split[TARGET]>0).mean()
    dist = split[TARGET].value_counts(normalize=True).sort_index().mul(100).round(1)
    print(f"{name:<12} rows={len(split):>7,}  positive={pos:.1f}%  {dict(dist)}")
 
assert set(TRAIN_YEARS).isdisjoint(VAL_YEARS)
assert set(TRAIN_YEARS).isdisjoint(TEST_YEARS)
assert set(VAL_YEARS).isdisjoint(TEST_YEARS)
assert set(STRESS_YEARS).isdisjoint(TEST_YEARS)
print("\nAll splits temporally disjoint")
 
# Save
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_model.to_parquet(OUT_DIR/"algeria_wildfire_engineered.parquet", index=False)
train.to_parquet(OUT_DIR/"train.parquet", index=False)
val.to_parquet(OUT_DIR/"val.parquet", index=False)
test.to_parquet(OUT_DIR/"test.parquet", index=False)
stress_20.to_parquet(OUT_DIR/"stress_2020.parquet", index=False)
 
feature_meta = {
    "target": TARGET,
    "final_features": FINAL_FEATURES,
    "id_cols": ID_COLS,
    "train_years": TRAIN_YEARS,
    "validation_years": VAL_YEARS,
    "test_years": TEST_YEARS,
    "stress_years": STRESS_YEARS,
    "n_features": len(FINAL_FEATURES),
    "n_train": len(train),
    "n_val": len(val),
    "n_test": len(test),
    "n_stress": len(stress_20),
}
with open(OUT_DIR/"feature_meta.json","w") as f:
    json.dump(feature_meta, f, indent=2)
 
print(f"\nSaved to {OUT_DIR}")
print(f"Features: {len(FINAL_FEATURES)}")


TRAIN        rows= 56,763  positive=19.0%  {0: np.float64(81.0), 1: np.float64(14.8), 2: np.float64(4.2)}
VAL          rows= 15,717  positive=28.1%  {0: np.float64(71.9), 1: np.float64(20.2), 2: np.float64(7.9)}
TEST         rows= 68,581  positive=15.7%  {0: np.float64(84.3), 1: np.float64(12.2), 2: np.float64(3.6)}
STRESS-2020  rows= 16,899  positive=33.1%  {0: np.float64(66.9), 1: np.float64(22.1), 2: np.float64(11.0)}

All splits temporally disjoint

Saved to ../data/training/engineered
Features: 37
